In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import os.path as osp

from pathlib import Path
from pprint import pprint

import sqlparse
import sys

root = Path.cwd().parent 
if str(root) not in sys.path:    
    sys.path.append(str(root))

In [3]:
import duckdb 

from src import logger
from src.config import CFG
from src.db.election_db import ElectionDB
from src.agent import RAGAgent, SQLAgent, HybridAgent, QueryIntent

/Users/dric/projects/research/LLMs/chat_app/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [4]:
agent = HybridAgent()

agent()

2026-03-08 20:33:44,340 [INFO] LocalLLMStack: Agent::Setting vllm_url='http://127.0.0.1:8000/v1'
2026-03-08 20:33:44,341 [INFO] LocalLLMStack: HybridAgent::Setting vllm_url='http://127.0.0.1:8000/v1'
2026-03-08 20:33:44,341 [INFO] LocalLLMStack: Agent::Setting vllm_url='http://127.0.0.1:8000/v1'
2026-03-08 20:33:44,352 [INFO] LocalLLMStack: SQLAgent::Setting vllm_url='http://127.0.0.1:8000/v1'
2026-03-08 20:33:44,361 [INFO] LocalLLMStack: Agent::Setting vllm_url='http://127.0.0.1:8000/v1'
2026-03-08 20:33:44,363 [INFO] LocalLLMStack: RAGAgent::Setting vllm_url='http://127.0.0.1:8000/v1'
2026-03-08 20:33:44,365 [INFO] LocalLLMStack: Agent Initialized


In [5]:
agent.client.streaming, agent.client.openai_api_base

(False, 'http://127.0.0.1:8000/v1')

### Inspecting tools

In [6]:
for idx, t in enumerate(agent.tools):
    print(f" {idx+1}) {t.name}: \t{t.description[:30]}...")

 1) describe_table: 	Get column names, types, and d...
 2) execute_read_query: 	Executes a SQL SELECT query an...
 3) get_db_schema: 	Returns the schema (table name...
 4) list_tables: 	Get a list of all available ta...
 5) sample_data: 	Fetch the first 3 rows of a ta...
 6) validate_sql: 	Validating SQL syntax and logi...
 7) search_database: 	Performs hybrid (Full-Text and...


## Check DB search operations

In [ ]:
agent.sql_expert.list_tables.args

In [ ]:
agent.sql_expert.list_tables.invoke({
    "reasoning": "just listing tables"
})

In [ ]:
agent.sql_expert.sample_data.args

In [ ]:
agent.sql_expert.sample_data.invoke({
    "reasoning": "sampling some data to see the values",
    "table_name": "vw_results",
    "num_samples": 10
})

In [ ]:
agent.sql_expert.describe_table.args

In [ ]:
agent.sql_expert.describe_table.invoke({
    "reasoning": "describing the table structure",
    "table_name": "vw_party"
})

### Vector search (VS)

In [ ]:
results = agent.db_client.vector_search(
    reasoning="performing some vector search",
    query="Who was elected in the Agneby-Tiassa region from RHDP?", 
    top_k=5
)

for text, chunk_id, score in results:
    print(f"chunk_id {chunk_id}: \t[{score:.2f}] {text}")

### Full text search (FTS)

In [ ]:
results = agent.db_client.full_text_search(
    query="What was the turnout in tiapoum?", 
    top_k=5
)

for text, chunk_id, score in results:
    print(f"chunk_id {chunk_id}: \t[{score:.2f}] {text}")

### Hybrid search

In [ ]:
results = agent.db_client.hybrid_search(
    query="Who was elected in Tiapoum region?", 
    top_k=5
)

for text, chunk_id, score in results:
    print(f"chunk_id {chunk_id}: \t[{score:.2f}] {text}")

## Check Agent-specific operations

### Testing Security Sandbox

In [ ]:
sql = "SELECT * FROM REGION"
parsed = sqlparse.parse(sql)

clean_sql = parsed[0]

clean_sql, clean_sql.value

In [ ]:
agent.sql_expert.validate_sql.args

In [ ]:
try:
    agent.forbidden = ["SELECT"] # Attempt to change the list
    agent._forbidden = [] # Attempt to change the list

except AttributeError as e:
    print(f"✅ Success: Modification blocked - {e}")

# Manual validation test
for kw in ["DROP", "ALTER", "UPDATE"]:
    malicious = f"SELECT * FROM results; {kw} TABLE turnout;"
    is_safe, out_message = agent.sql_expert.validate_sql.invoke({
        "reasoning": "validating SQL query",
        "sql": malicious,
        "metrics": agent.sql_expert.metrics,
        "forbidden": agent.sql_expert.forbidden
    })
    print(f"Is {kw} query safe? {is_safe} (Expected: False)")
    print(f"Message: {out_message}")
    print()

for kw in ["DROP", "ALTER", "UPDATE"]:
    malicious = f"{kw} TABLE result;"
    is_safe, out_message = agent.sql_expert.validate_sql.invoke({
        "reasoning": "validating SQL query",
        "sql": malicious,
        "metrics": agent.sql_expert.metrics,
        "forbidden": agent.sql_expert.forbidden
    })
    print(f"Is {kw} query safe?\t {is_safe} (Expected: False)")
    print(f"Message: {out_message}")
    print()

In [ ]:
is_safe, out_message = agent.sql_expert.validate_sql.invoke({
    "reasoning": "validating SQL query",
    "sql": clean_sql.value,
    "metrics": agent.sql_expert.metrics,
    "forbidden": agent.sql_expert.forbidden
})
print(f"Is query safe? {is_safe} (Expected: False)")
print(f"Message: {out_message}")
print()

In [ ]:
is_valid, msg = agent.sql_expert.validate_sql.invoke({
    "sql": "SELECT PARTY_NAME, SUM(SCORES) AS total_votes FROM result GROUP BY PARTY_ID ORDER BY total_votes DESC LIMIT 20",
    "reasoning": "validating SQL query",
    "metrics": agent.sql_expert.metrics,
    "forbidden": agent.sql_expert.forbidden
})

print(f"Is query safe? {is_safe} (Expected: False)")
print(f"Message: {out_message}")

In [ ]:
agent.sql_expert.execute_read_query.args

In [ ]:
results, out_msg = agent.sql_expert.execute_read_query.invoke({
    "sql_query": "SELECT PARTY_NAME, SUM(SCORES) AS total_votes FROM result GROUP BY PARTY_ID ORDER BY total_votes DESC LIMIT 20",
    "reasoning": "checking query"
})

print(f"{results=}")
print(f"{out_msg=}")

### Testing intent routing

In [ ]:
test_queries = [
    ("Who got the most votes in Abidjan?", "RANKING"),
    ("What is the total turnout?", "AGGREGATION"),
    ("How many colors are there in the rainbow?", "INVALID"),
    ("Tell me about the history of the 2020 election.", "GENERAL"),
    ("What is the weather in Paris?", "INVALID")
]

for query, expected in test_queries:
    intent = agent._get_intent(query)
    print(f"Query: {query} \n> Detected: {intent} (Expected: {expected})\n")
    print()


### SQLAgent 

In [ ]:
# from langchain_openai import ChatOpenAI
# import sys

# # Test a raw stream
# for chunk in CFG.client.stream("Write a 2-sentence story about PhD."):
#     sys.stdout.write(chunk.content)
#     sys.stdout.flush()


In [ ]:
%%time 

query = "What is the total votes by party ?"
intent=QueryIntent.AGGREGATION

if CFG.IS_STREAM:
    print(f"--- Starting Stream for: {query} ---\n")
    
    for chunk in agent.sql_expert.generate_sql(query, intent):
        if chunk["type"] == "status":
            print(f"\n\n[STATUS]: {chunk['content']}", end="", flush=True)
        
        elif chunk["type"] == "token":
            print(chunk["content"], end="", flush=True)
            
        elif chunk["type"] == "action":
            # Visual separator for tool calls
            print(f"\n\n🛠️  [SYSTEM]: {chunk['content']}", end="", flush=True)
            
        elif chunk["type"] == "final_sql":
            print(f"\n\n🚀 [FINAL SQL]:\n{chunk['content']}", end="", flush=True)
            final_sql = chunk['content']
            print(f"\n\n🚀 [SUCCESS]: SQL Captured.")
    
        elif chunk["type"] == "error":
            print(f"\n❌ [ERROR]: {chunk['content']}", end="", flush=True)

else:
    response = agent.sql_expert.generate_sql(query, intent)

In [ ]:
agent.generation_prompt

In [ ]:
pprint(response)

In [ ]:
results, out_msg = agent.sql_expert.execute_read_query.invoke({
    "sql_query": """
                SELECT PARTY_NAME, SUM(TOTAL_VOTES) AS TOTAL_VOTES 
                FROM vw_party 
                GROUP BY PARTY_NAME 
                ORDER BY TOTAL_VOTES DESC 
                LIMIT 100;
                """,
    "reasoning": "checking query"
})

print(f"{results.shape=}")
print(f"{out_msg=}")

intent = agent._get_intent(user_prompt=query)

out = agent._interpret_results(
    user_prompt=query,
    df=results,
    intent=intent
)

pprint(out) 

In [ ]:
query = "Which region has the most voters?"
intent = QueryIntent.RANKING  # Manually set for the test
final_sql_query = None

if CFG.IS_STREAM:
    print(f"--- Starting Stream for: {query} ---\n")
    
    for chunk in agent.sql_expert.generate_sql(query, intent):
        
        if chunk["type"] == "token":
            print(chunk["content"], end="", flush=True)
            
        elif chunk["type"] == "action":
            print(f"\n\n🛠️  [ACTION]: {chunk['content']}")
            
        elif chunk["type"] == "final_sql":
            print(f"\n\n🚀 [FINAL SQL]:\n{chunk['content']}")
            final_sql = chunk['content']
    
    print("\n--- Stream Finished ---")
else:
    response = agent.sql_expert.generate_sql(query, intent)

In [ ]:
response

In [ ]:
query = "What is the turnout in the region with the most voters?"
intent = QueryIntent.AGGREGATION 

if CFG.IS_STREAM:

    print("--- [STARTING STREAM] ---")
    for update in agent.sql_expert.generate_sql(query, intent):
        
        if update["type"] == "token":
            print(update["content"], end="", flush=True)
            
        elif update["type"] == "action":
            print(f"\n\n🛠️  [TOOL CALL]: {update['content']}")
            
        elif update["type"] == "final_sql":
            print(f"\n\n🚀 [FINAL SQL]:\n{update['content']}")
    
    print("\n--- [STREAM FINISHED] ---")
else:
    response = agent.sql_expert.generate_sql(query, intent)

### Full End-to-End Analytical Run

In [7]:
vague_query = "Who won the elections in Abidjan?" 
# Logic: Abidjan is a wide region so agent might need clarification

agent._get_intent(user_prompt=vague_query)

2026-03-08 20:33:46,841 [INFO] LocalLLMStack: Formatting input to LLM
2026-03-08 20:33:56,996 [INFO] httpx: HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


<QueryIntent.RANKING: 'ranking'>

In [8]:
agent.rag_expert.search_database.args

{'query': {'title': 'Query', 'type': 'string'},
 'db_client': {'title': 'Db Client'}}

In [9]:
results = agent.rag_expert.search_database.invoke({    
    "query": vague_query,
    "db_client": agent.db_client
    })

2026-03-08 20:34:23,919 [INFO] sentence_transformers.SentenceTransformer: Use pytorch device_name: mps
2026-03-08 20:34:23,920 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: google/embeddinggemma-300m
2026-03-08 20:34:24,470 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/modules.json "HTTP/1.1 200 OK"
2026-03-08 20:34:24,614 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-08 20:34:24,766 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-08 20:34:24,905 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/README.md "HTTP/1.1 200 OK"
2026-03-08 20:34:25,044 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/m

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

2026-03-08 20:34:26,163 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-08 20:34:26,312 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-08 20:34:26,475 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/models/google/embeddinggemma-300m/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-08 20:34:26,612 [INFO] httpx: HTTP Request: GET https://huggingface.co/api/models/google/embeddinggemma-300m/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-08 20:34:28,871 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/1_Pooling/config.json "HTTP/1.1 200 OK"
2026-03-08 20:34:29,047 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/2_Dense/config.json "HTTP/1.1 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
for text, chunk_id, score in results:
    print(f"chunk_id {chunk_id}: \t[{score:.2f}] {text}")

chunk_id 173: 	[0.02] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, ABOBO RENAISSANCE (INDEPENDANT) received 916 votes (0.87%) but has lost.
chunk_id 174: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, AGIR AUJOURDHUI POUR BATIR DEMAIN (AIDE) received 1374 votes (1.31%) but has lost.
chunk_id 1125: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, UNE CÔTE DIVOIRE EN PAIX, PROSPERE ET SOLIDAIRE (RHDP) received 92947 votes (88.45%) but has lost.
chunk_id 1127: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, TOUS ENSEMBLE POUR LA CÔTE D'IVOIRE (PDCI-RDA) received 7359 votes (7.0%) but has lost.
chunk_id 178: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, TOUS ENSEMBLE POUR LA CÔTE D'IVOIRE (PDCI-RDA) received 7359 votes (7.0%) but has lost.
chunk_id 1439: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED

### get_answer()

In [11]:
print(f"--- Testing Disambiguation for: {vague_query} ---")
try:
    for response in agent.get_answer(vague_query):

        if response.get("type") == "clarification":
            print("Success: Agent identified ambiguity.")
            print("Agent Question:", response["content"])
            print("Options found in DB:", response["options"])

except Exception as e:
    logger.error(f"Failure: Agent failed to process query. {e}", exc_info=True)


--- Testing Disambiguation for: Who won the elections in Abidjan? ---
2026-03-08 20:35:07,076 [INFO] LocalLLMStack: Formatting input to LLM
2026-03-08 20:35:11,020 [INFO] httpx: HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-08 20:35:11,024 [INFO] LocalLLMStack: Identifying decision route using LLM routing...
2026-03-08 20:35:18,879 [INFO] httpx: HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-08 20:35:18,891 [INFO] LocalLLMStack: ✅ Identified route: RAG
2026-03-08 20:35:37,253 [INFO] httpx: HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-08 20:35:37,255 [INFO] LocalLLMStack: Extracting tool calls
2026-03-08 20:35:37,256 [INFO] LocalLLMStack: Executing tool: search_database with args {'query': 'Who won the elections in Abidjan?', 'db_client': 'elections'}


In [13]:
response

{'type': 'status',
 'content': "Could not retrieve response. 'str' object has no attribute 'hybrid_search'"}

#### Test various queries

In [ ]:
queries = [
    "How many seats did RHDP win?",
    "Who won the elections in tiapum",
    "Top 10 candidates by score in region Nawa.",
    "Participation rate by region",
    "Histogram of winners by party.",
    "distribution of winners per party",
    "Which party did win the most seats?",
    "Show me the distribution of voters per party and per region"
]

for query in queries:
    print(f'>> Query: {query}')
    out = agent.get_answer(user_prompt=query)
    print(f'>> Response: {out["content"]}\n')
    break
